# Visor 3D interactivo de DaT scans

Este notebook abre **un archivo NIfTI a la vez directamente desde `data/raw/niftis.zip`**, sin modificar la fuente original. La visualización usa Plotly/WebGL.

- Arrastra con el botón izquierdo para rotar.
- Usa la rueda para acercar o alejar.
- Arrastra con el botón derecho para desplazar la escena.
- Usa la barra superior del gráfico para restablecer la cámara o guardar una imagen.

> **Privacidad:** ejecútalo sólo localmente. No publiques el notebook con salidas guardadas ni subas imágenes o datos de la competencia a servicios externos.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import ipywidgets as widgets
import nibabel as nib
import numpy as np
import plotly.graph_objects as go
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

ARCHIVE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'niftis.zip'
if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f'No se encontró el archivo esperado: {ARCHIVE_PATH}')

with ZipFile(ARCHIVE_PATH) as archive:
    NIFTI_MEMBERS = sorted(
        name for name in archive.namelist()
        if name.lower().endswith(('.nii', '.nii.gz')) and not name.endswith('/')
    )

if not NIFTI_MEMBERS:
    raise RuntimeError('El ZIP no contiene archivos .nii o .nii.gz.')

print(f'Archivo fuente: {ARCHIVE_PATH.name}')
print(f'Escaneos disponibles: {len(NIFTI_MEMBERS):,}')

Archivo fuente: niftis.zip
Escaneos disponibles: 1,362


In [2]:
def load_nifti_from_zip(member_name: str) -> tuple[np.ndarray, tuple[float, float, float]]:
    """Extrae temporalmente un NIfTI, lo orienta y devuelve volumen y espaciado."""
    with TemporaryDirectory(prefix='dat_scan_') as temp_dir:
        with ZipFile(ARCHIVE_PATH) as archive:
            extracted_path = Path(archive.extract(member_name, path=temp_dir))
        image = nib.as_closest_canonical(nib.load(extracted_path))
        volume = np.asarray(image.dataobj, dtype=np.float32).squeeze()
        spacing = tuple(float(value) for value in image.header.get_zooms()[:3])

    if volume.ndim != 3:
        raise ValueError(f'Se esperaba un volumen 3D y se obtuvo shape={volume.shape}.')
    return volume, spacing


def robust_normalize(volume: np.ndarray) -> np.ndarray:
    finite = volume[np.isfinite(volume)]
    if finite.size == 0:
        raise ValueError('El volumen no contiene valores finitos.')

    positive = finite[finite > 0]
    reference = positive if positive.size else finite
    low, high = np.percentile(reference, [1.0, 99.5])
    if high <= low:
        raise ValueError('El volumen no tiene rango de intensidades suficiente.')

    normalized = (np.nan_to_num(volume, nan=low) - low) / (high - low)
    return np.clip(normalized, 0.0, 1.0)


def downsample_volume(
    volume: np.ndarray,
    spacing: tuple[float, float, float],
    max_side: int,
) -> tuple[np.ndarray, tuple[float, float, float], tuple[int, int, int]]:
    steps = tuple(max(1, int(np.ceil(size / max_side))) for size in volume.shape)
    sampled = volume[::steps[0], ::steps[1], ::steps[2]]
    sampled_spacing = tuple(spacing[i] * steps[i] for i in range(3))
    return sampled, sampled_spacing, steps


def build_figure(
    volume: np.ndarray,
    spacing: tuple[float, float, float],
    member_name: str,
    mode: str,
    threshold: float,
    opacity: float,
) -> go.Figure:
    x = np.arange(volume.shape[0], dtype=np.float32) * spacing[0]
    y = np.arange(volume.shape[1], dtype=np.float32) * spacing[1]
    z = np.arange(volume.shape[2], dtype=np.float32) * spacing[2]
    xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')

    common = dict(
        x=xx.ravel(),
        y=yy.ravel(),
        z=zz.ravel(),
        value=volume.ravel(),
        isomin=threshold,
        isomax=1.0,
        colorscale='Turbo',
        colorbar=dict(title='Intensidad normalizada'),
    )

    if mode == 'Volumen':
        trace = go.Volume(
            **common,
            opacity=opacity,
            surface_count=20,
        )
    else:
        trace = go.Isosurface(
            **common,
            opacity=opacity,
            surface_count=5,
            caps=dict(x_show=False, y_show=False, z_show=False),
        )

    figure = go.Figure(trace)
    figure.update_layout(
        title=f'{Path(member_name).name} · {mode}',
        height=760,
        margin=dict(l=0, r=0, t=55, b=0),
        scene=dict(
            aspectmode='data',
            dragmode='orbit',
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            camera=dict(projection=dict(type='perspective')),
        ),
    )
    return figure

In [3]:
file_selector = widgets.Dropdown(
    options=[(Path(name).name, name) for name in NIFTI_MEMBERS],
    description='Escaneo:',
    layout=widgets.Layout(width='650px'),
    style={'description_width': '90px'},
)
mode_selector = widgets.ToggleButtons(
    options=['Volumen', 'Isosuperficie'],
    value='Volumen',
    description='Modo:',
    style={'description_width': '90px'},
)
resolution_selector = widgets.IntSlider(
    value=96, min=48, max=144, step=16,
    description='Resolución:', continuous_update=False,
    style={'description_width': '90px'},
)
threshold_selector = widgets.FloatSlider(
    value=0.20, min=0.02, max=0.80, step=0.02,
    description='Umbral:', continuous_update=False, readout_format='.2f',
    style={'description_width': '90px'},
)
opacity_selector = widgets.FloatSlider(
    value=0.12, min=0.03, max=0.80, step=0.03,
    description='Opacidad:', continuous_update=False, readout_format='.2f',
    style={'description_width': '90px'},
)
render_button = widgets.Button(
    description='Renderizar 3D',
    button_style='primary',
    icon='cube',
)
output = widgets.Output()


def render_selected(_button: widgets.Button) -> None:
    with output:
        output.clear_output(wait=True)
        member_name = file_selector.value
        print(f'Cargando localmente: {Path(member_name).name} ...')
        try:
            raw_volume, spacing = load_nifti_from_zip(member_name)
            normalized = robust_normalize(raw_volume)
            sampled, sampled_spacing, steps = downsample_volume(
                normalized, spacing, resolution_selector.value
            )
            figure = build_figure(
                sampled,
                sampled_spacing,
                member_name,
                mode_selector.value,
                threshold_selector.value,
                opacity_selector.value,
            )
            display(Markdown(
                f'**Shape original:** `{raw_volume.shape}` · '
                f'**Shape mostrada:** `{sampled.shape}` · '
                f'**Submuestreo:** `{steps}`'
            ))
            figure.show(config={
                'scrollZoom': True,
                'displayModeBar': True,
                'responsive': True,
            })
        except Exception as error:
            print(f'No fue posible renderizar el volumen: {error}')


render_button.on_click(render_selected)
controls = widgets.VBox([
    file_selector,
    mode_selector,
    resolution_selector,
    threshold_selector,
    opacity_selector,
    render_button,
])
display(controls, output)

Output()

## Consejos de uso

- Comienza con resolución **96**. Auméntala sólo si el movimiento sigue fluido.
- En **Volumen**, baja el umbral para incluir más señal y aumenta la opacidad gradualmente.
- En **Isosuperficie**, sube el umbral para destacar las regiones de mayor captación.
- Para liberar memoria, selecciona otro escaneo y vuelve a renderizar; la salida anterior se reemplaza.
- Antes de compartir o versionar el notebook, usa **Clear All Outputs**. Las salidas 3D contienen intensidades derivadas del escaneo.